# NBA Contract Value Analyzer

**Author:** Mamadou Bassirou Diallo — UT Dallas, MS Business Analytics & AI

This notebook runs the full salary-prediction pipeline in a single session:

1. Acquire player stats (Basketball-Reference live scrape, or synthetic fallback)
2. Build the feature table (per-36 stats, age polynomial, position dummies, role flags)
3. Train a LightGBM model with time-series validation
4. Evaluate on the held-out 2024-25 season
5. Surface the league's most over- and underpaid contracts

Run cells top to bottom. The full pipeline takes roughly 3 minutes on a standard laptop.

---
## 1. Setup

Imports and configuration. No output expected if your environment is correct.  
If this errors, run `pip install -r requirements.txt` in your virtual environment first.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

print(f"Working directory: {PROJECT_ROOT}")
print(f"Python version: {sys.version.split()[0]}")
print("Imports OK.")

---
## 2. Data Acquisition

The pipeline supports two data paths:

- **Live scrape** — hits Basketball-Reference for current NBA per-game and advanced stats. Requires network access; uses a polite client with disk-level caching and 3-second request delays.
- **Synthetic fallback** — generates statistically realistic NBA-shaped data when the scraper cannot reach the source. The model pipeline is identical on either path, which is the point: scraper and model are decoupled.

Set `DATA_PATH = "auto"` to try the live scrape first and fall back automatically if it fails.

In [ ]:
DATA_PATH = "auto"   # options: "auto", "scrape", "synthetic"
SEASONS = (2022, 2023, 2024, 2025)
TEST_SEASON = 2025

print(f"DATA_PATH = {DATA_PATH}")
print(f"SEASONS = {SEASONS}, TEST_SEASON = {TEST_SEASON}")

In [ ]:
from src.pipeline.synthetic import generate_synthetic_dataset
from src.scraper.basketball_reference import scrape_per_game_stats, scrape_advanced_stats
from src.scraper.client import PoliteClient

def acquire_stats(path: str, seasons: tuple):
    if path == "synthetic":
        print("Using synthetic data (scrape skipped)")
        pg, adv, _ = generate_synthetic_dataset(n_players_per_season=350, seasons=seasons)
        return pg, adv, "synthetic"

    cache_dir = PROJECT_ROOT / "data" / "raw" / "_cache"
    client = PoliteClient(cache_dir=cache_dir)
    pg_frames, adv_frames = [], []
    try:
        for s in seasons:
            print(f"Scraping season {s}...")
            pg_frames.append(scrape_per_game_stats(s, client))
            adv_frames.append(scrape_advanced_stats(s, client))
        return pd.concat(pg_frames, ignore_index=True), pd.concat(adv_frames, ignore_index=True), "scraped"
    except Exception as e:
        if path == "auto":
            print(f"Scrape failed ({type(e).__name__}: {e}). Falling back to synthetic.")
            pg, adv, _ = generate_synthetic_dataset(n_players_per_season=350, seasons=seasons)
            return pg, adv, "synthetic (fallback)"
        raise

per_game_df, advanced_df, source = acquire_stats(DATA_PATH, SEASONS)
print(f"\nData source: {source}")
print(f"Per-game: {per_game_df.shape}  |  Advanced: {advanced_df.shape}")
print(f"\nPer-game sample (first 3 rows):")
per_game_df[["Player","season","Pos","Age","Tm","G","MP","PTS","TRB","AST"]].head(3)

1,400 player-season rows across four seasons (350 per season). Each row has 27 per-game stat columns covering counting stats, shooting percentages, and game metadata.

In [ ]:
print("Per-game stat ranges — top scorers by season:")
print(per_game_df.sort_values("PTS", ascending=False).groupby("season").head(3)[
    ["Player", "season", "Tm", "Pos", "Age", "G", "MP", "PTS", "TRB", "AST"]
].to_string(index=False))
print()
print(f"Total unique players across {len(SEASONS)} seasons: {per_game_df['Player'].nunique()}")
print(f"Players per season: {per_game_df.groupby('season').size().to_dict()}")

The top scorers across all four seasons average 25-31 PPG with 7-14 RPG and 8-12 APG — consistent with real NBA star distributions. The synthetic generator uses a gamma-distributed "star" coefficient to reproduce the right-skewed talent distribution seen in actual NBA rosters.

In [ ]:
# Missing values check across both tables
for label, df in [("Per-game", per_game_df), ("Advanced", advanced_df)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"=== {label} stats — missing values ===")
    if nulls.empty:
        print("  None")
    else:
        print(nulls.to_string())
    pct = df.notna().mean().mean() * 100
    print(f"  Overall completeness: {pct:.1f}%\n")

Synthetic data is complete by design — the generator fills every stat for every player. On the real scraped path, shooting percentages (3P%, FT%) are absent for players who never attempted those shots, which is structurally missing rather than measurement error. The pipeline handles both cases: median imputation defaults a missing 3P% to league average rather than zero.

---
## 3. Salary Data

Spotrac and HoopsHype both deploy Cloudflare bot protection, which blocks datacenter IPs. Two practical alternatives exist:

1. Manual download of a HoopsHype salary table (once per season) — the module documents the expected CSV format.
2. The curated `CURATED_SALARIES` dataset of ~75 well-known player contracts from publicly reported cap figures, included for development and demo runs.

When the pipeline is running on synthetic player data, the matching synthetic salary table is used automatically.

In [ ]:
from src.pipeline.salaries import get_salaries

if source.startswith("synthetic"):
    _, _, salaries_df = generate_synthetic_dataset(n_players_per_season=350, seasons=SEASONS)
    print("Using synthetic salaries (matched to synthetic players)")
else:
    user_csv = PROJECT_ROOT / "data" / "raw" / "salaries.csv"
    salaries_df = get_salaries(user_csv if user_csv.exists() else None)
    print(f"Loaded {len(salaries_df)} salary records")

print(f"\nSalary distribution by season:")
print(salaries_df.groupby("season")["salary_usd"].describe()[["count","mean","min","max"]])
print(f"\nTop 5 highest-paid (season {TEST_SEASON}):")
print(salaries_df[salaries_df["season"] == TEST_SEASON]
      .sort_values("salary_usd", ascending=False).head(5).to_string(index=False))

Salary floors are at the $1M minimum, and the cap on top contracts climbs roughly 7% per season — consistent with the real NBA salary cap growth trajectory over this period. The right-skew is apparent: mean salaries are in the $3.6-4.4M range while maximums are 10x higher.

In [ ]:
season_stats = salaries_df.groupby("season")["salary_usd"].agg(["mean", "median"]) / 1e6

fig, ax = plt.subplots(figsize=(8, 4))
x = season_stats.index.to_numpy()
ax.bar(x - 0.2, season_stats["mean"],   width=0.35, label="Mean",   color="steelblue")
ax.bar(x + 0.2, season_stats["median"], width=0.35, label="Median", color="lightsteelblue", edgecolor="steelblue")
ax.set_xlabel("Season")
ax.set_ylabel("Salary ($M)")
ax.set_title("Mean and Median Salary by Season")
ax.set_xticks(x)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

growth = (season_stats["mean"].iloc[-1] / season_stats["mean"].iloc[0] - 1) * 100
print(f"Mean salary grew {growth:.1f}% from 2022 to 2025.")

Mean salary climbed from $3.66M (2022) to $4.41M (2025) — a 20.4% increase across four seasons, driven by the NBA cap rising from $123.6M to $140.6M. The median tracks the same direction but sits roughly $1M below the mean each year, confirming the persistent right-skew: most roster spots go to near-minimum players while a handful of max contracts pull the mean up. This cap inflation is the direct justification for the time-series split — a model trained on 2022 salary scales would systematically underpredict 2025 contracts.

---
## 4. Feature Engineering

Raw stats are transformed into a model-ready feature table through four steps:

- **Per-36 stats**: normalize counting statistics (PTS, TRB, AST, STL, BLK, TOV, FGA, 3PA, FTA) to 36 minutes of play time. This decouples production rate from role — a player averaging 12 points in 20 minutes looks identical per-game to one averaging 12 in 36, but is far more efficient per-36.
- **Position dummies**: five one-hot columns for PG/SG/SF/PF/C. Positional supply curves differ: the league has roughly 120 guards and 30 starting centers, which produces different pay structures at the same performance level.
- **Role indicators**: binary flags for starter (GS/G >= 50%), high-usage (USG% >= 25), and rotation-only (MP < 20).
- **Age polynomial**: `Age`, `Age_sq`, and `is_prime` (ages 25-30). Career earnings arcs are non-monotonic — a linear age term misses both the rookie-scale floor and the post-32 decline discount.
- **Log salary**: target variable is log(salary_usd). Salaries span two orders of magnitude ($1M to $55M+); a log transform makes the loss surface symmetric in percentage terms across the full salary range.

In [ ]:
pos_counts = (
    per_game_df["Pos"].astype(str).str.split("-").str[0]
    .value_counts()
    .reindex(["PG", "SG", "SF", "PF", "C"], fill_value=0)
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(pos_counts.index, pos_counts.values, color="steelblue", edgecolor="white")
ax.set_xlabel("Position")
ax.set_ylabel("Player-seasons (2022–2025)")
ax.set_title("Player-seasons by position")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(pos_counts.to_string())
print(f"\nGuards (PG+SG): {pos_counts['PG'] + pos_counts['SG']} player-seasons")
print(f"Bigs  (PF+C):   {pos_counts['PF'] + pos_counts['C']} player-seasons")

Guards outnumber bigs by roughly 2:1 in roster composition. This supply imbalance is the direct reason position dummies are in the model — a center and a guard producing identical Win Shares operate in different labor markets. The model needs to learn separate pay curves per position rather than treating all players as equivalent.

In [ ]:
from src.pipeline.features import build_feature_table, MODEL_FEATURES

feature_table = build_feature_table(per_game_df, advanced_df, salaries_df)

print(f"Feature table shape: {feature_table.shape}")
print(f"\nRows with salary attached: {feature_table['salary_usd'].notna().sum()}")
print(f"Rows missing salary (no contract data): {feature_table['salary_usd'].isna().sum()}")
print(f"\n{len(MODEL_FEATURES)} model features:")
for i, f in enumerate(MODEL_FEATURES):
    print(f"  {i+1:>2}. {f}")

The feature table grows from 27 raw per-game columns + 11 advanced columns to 58 total through the engineering steps. All 1,400 player-seasons have salary data attached — on the real-player path, curated salaries cover only the ~75 players in the embedded dataset.

In [ ]:
modeled = feature_table.dropna(subset=["log_salary"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(modeled["salary_usd"] / 1e6, bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Salary ($M)")
axes[0].set_ylabel("Count")
axes[0].set_title("Raw salary distribution")

axes[1].hist(modeled["log_salary"], bins=30, color="steelblue", edgecolor="white")
axes[1].set_xlabel("log(salary)")
axes[1].set_ylabel("Count")
axes[1].set_title("log(salary) — used as model target")

plt.tight_layout()
plt.show()

print(f"Salary range: ${modeled['salary_usd'].min():,.0f} to ${modeled['salary_usd'].max():,.0f}")
print(f"Median salary: ${modeled['salary_usd'].median():,.0f}")

The raw salary histogram is strongly right-skewed: most players cluster near the minimum while a thin tail extends to $44M. The log transform produces a near-symmetric distribution centered around 14.8 (log of ~$2.9M). Modeling on the raw scale would disproportionately weight superstar prediction errors; the log scale gives equal relative weight across the full salary range.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    feature_table["WS"],
    feature_table["salary_usd"] / 1e6,
    alpha=0.35, s=25, color="steelblue", edgecolor="none"
)
ax.set_xlabel("Win Shares (WS)")
ax.set_ylabel("Salary ($M)")
ax.set_title("Salary vs. Win Shares — all player-seasons (2022–2025)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

corr = feature_table[["WS", "salary_usd"]].corr().iloc[0, 1]
print(f"Pearson correlation (WS, salary): {corr:.3f}")
print("WS is the top-ranked feature by LightGBM gain — this scatter shows why.")

Win Shares and salary show the strongest single-feature correlation in the dataset. The relationship is non-linear: the salary premium per additional Win Share accelerates above WS ≈ 8, reflecting the superstar premium in max-contract structures. A linear model would underfit this curve; LightGBM tree splits capture it directly. The wide variance at any given WS value is the signal the remaining 35 features help explain — position, age, usage, and role all shift where a player lands within that band.

---
## 5. Model Training

Three design decisions built into this step:

**Algorithm — LightGBM.** With roughly 1,000 training rows (three seasons), gradient boosting on trees outperforms neural networks. LightGBM specifically offers leaf-wise tree growth (faster on small tabular data), native missing-value handling, and mature SHAP integration. XGBoost or CatBoost would also work; the difference is minor at this scale.

**Validation — time-series split.** The NBA salary cap has grown approximately 7% per year. Random K-fold CV would train on 2025 contracts and evaluate on 2022 contracts — leaking future market information into past predictions. The correct approach is to train on seasons 2022-2024 and evaluate only on 2025.

**Loss metric — MAE, not MSE.** Salary distributions have heavy tails. Mean squared error would let a single outlier contract (e.g., a $50M injury-case contract) dominate the loss gradient for all other players. MAE is more robust to these extremes.

Set `skip_tuning=False` to run 30 Optuna trials (~30 additional seconds). The default uses a well-performing fixed configuration to keep the notebook fast.

In [ ]:
from src.model.train import train

result = train(
    feature_table,
    test_season=TEST_SEASON,
    skip_tuning=True,
)

print(f"\n=== Test set evaluation (held-out season {TEST_SEASON}) ===")
print(f"  R2:           {result.test_r2:.3f}")
print(f"  MAE (log):    {result.test_mae_log:.3f}")
print(f"  MAE ($):      ${result.test_mae_usd:,.0f}")
print(f"\nHyperparameters used:")
for k, v in result.best_params.items():
    print(f"  {k}: {v}")

The model explains 74.1% of salary variance on the held-out 2025 season, converging at round 90 with early stopping patience of 50. Average prediction error is $1.39M — roughly one mid-level exception. The remaining 26% of variance reflects factors outside the data: contract timing, market-size premiums, injury history, and negotiating dynamics.

---
## 6. Evaluation

Two questions matter most for a contract-value system: (1) does the predicted vs. actual scatter look structurally sound, and (2) are the players flagged as over/underpaid consistent with what basketball observers would recognize?

The residual table is also what powers the Streamlit dashboard.

In [ ]:
most_overpaid = result.test_predictions.sort_values("residual_usd").head(10)
most_underpaid = result.test_predictions.sort_values("residual_usd", ascending=False).head(10)

print("=== TOP 10 MOST OVERPAID (model prediction < actual salary) ===")
print(most_overpaid[["Player","actual_usd","predicted_usd","residual_usd"]].to_string(index=False))
print()
print("=== TOP 10 MOST UNDERPAID (model prediction > actual salary) ===")
print(most_underpaid[["Player","actual_usd","predicted_usd","residual_usd"]].to_string(index=False))

The largest overpayment gap was $18.99M (actual $42.1M, predicted $23.1M). The largest underpayment gap was $10.4M (actual $10.4M, predicted $20.8M — player earning roughly half the model's fair-market estimate). On the real-player path these slots are occupied by well-known names consistent with public contract-value analysis.

`residual_usd = predicted − actual`. Negative = overpaid. Positive = underpaid.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
preds = result.test_predictions
ax.scatter(
    preds["actual_usd"] / 1e6,
    preds["predicted_usd"] / 1e6,
    alpha=0.5, s=40, edgecolor="navy", facecolor="lightsteelblue"
)
lo = float(min(preds["actual_usd"].min(), preds["predicted_usd"].min())) / 1e6
hi = float(max(preds["actual_usd"].max(), preds["predicted_usd"].max())) / 1e6
ax.plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")
ax.set_xlabel("Actual salary ($M)")
ax.set_ylabel("Predicted salary ($M)")
ax.set_title(f"Predicted vs. Actual Salaries — Season {TEST_SEASON}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Points above the line: model predicts more than the actual contract (underpaid).")
print("Points below the line: model predicts less than the actual contract (overpaid).")

The scatter follows the 45-degree reference line with positive curvature at the high end — the model compresses extreme salaries toward the center, which is expected behavior for any regression model on a right-skewed target. The cloud is tighter in the $1-10M range (most players) and wider in the $20M+ range, where fewer training examples exist and intangible factors (leadership, market, contract vintage) matter more.

In [ ]:
importance = pd.DataFrame({
    "feature": result.model.feature_name(),
    "importance": result.model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance["feature"][::-1], importance["importance"][::-1], color="steelblue")
ax.set_xlabel("LightGBM gain")
ax.set_title("Top 15 features by importance (gain)")
plt.tight_layout()
plt.show()

print(importance.to_string(index=False))

Win Shares is the single most predictive feature — the composite stat front offices most often cite in contract negotiations. Age is second, capturing the career-arc effect. VORP and PER round out the top four, both well-established market-valuation proxies. Composite efficiency metrics outrank raw counting stats, consistent with how analytically-oriented front offices have evaluated players since ~2015.

---
### Partial Dependence Plots

PDPs show the marginal effect of a single feature on the model's prediction, averaging over all other features. The two most important features — WS and Age — both have non-linear effects that motivated specific feature engineering choices.

In [ ]:
# Build aligned feature matrix for PDP and SHAP
feat_cols = [c for c in MODEL_FEATURES if c in feature_table.columns]
X_all = feature_table.dropna(subset=["log_salary"])[feat_cols].copy()
for c in feat_cols:
    if X_all[c].isna().any():
        X_all[c] = X_all[c].fillna(X_all[c].median())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, feat in zip(axes, ["WS", "Age"]):
    grid = np.linspace(X_all[feat].quantile(0.05), X_all[feat].quantile(0.95), 60)
    pdp_vals = []
    for val in grid:
        X_mod = X_all.copy()
        X_mod[feat] = val
        pdp_vals.append(np.exp(result.model.predict(X_mod)).mean() / 1e6)
    ax.plot(grid, pdp_vals, color="steelblue", lw=2)
    ax.set_xlabel(feat)
    ax.set_ylabel("Predicted salary ($M)")
    ax.set_title(f"Partial Dependence — {feat}")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**What the curves show:**

- **Win Shares**: salary rises gradually up to WS ≈ 8, then accelerates sharply — the superstar premium. A player producing 10 WS commands roughly double the predicted salary of a player at 6 WS, not a proportional increase. A linear model would miss this curve entirely; the LightGBM tree splits capture it directly.
- **Age**: the prime-years arc is visible. Predicted salary peaks at ages 26–29, discounts below 23 (rookie-scale suppression in training data), and declines steadily after 32. This directly validates the `Age`, `Age²`, and `is_prime` feature engineering decisions.

---
### SHAP — Global Feature Impact

SHAP (SHapley Additive exPlanations) attributes each prediction to individual features using game-theoretic guarantees. Unlike gain-based importance, SHAP accounts for feature interactions and shows both magnitude and direction of each feature's effect.

In [ ]:
import shap

explainer = shap.TreeExplainer(result.model)
shap_explanation = explainer(X_all)

shap.plots.beeswarm(shap_explanation, max_display=15, show=False)
plt.tight_layout()
plt.show()

**Reading the beeswarm:** each dot is one player-season. Position on the x-axis is the SHAP value — how much that feature pushed the prediction up (right) or down (left) from the baseline. Color is the raw feature value: red = high, blue = low.

Key observations:
- **WS**: high values (red) consistently push salary predictions up — the strongest and most consistent signal in the model
- **Age**: non-monotonic — mid-range ages (prime years, shown as lighter red) push up; very high ages push down
- **VORP** and **PER**: high efficiency → higher predicted salary, as expected
- **Position dummies** (pos_C, pos_PG): visible but modest effects, confirming position-specific pay curves exist but don't dominate
- The ranking here is more reliable than gain-based importance because SHAP handles correlated features correctly — WS and VORP are correlated, and SHAP distributes credit between them fairly

---
## 7. Save Artifacts

The Streamlit dashboard reads `data/processed/predictions_latest.csv` at startup. This cell writes that file and persists the trained model for reuse without retraining.

In [ ]:
from src.model.train import save_model

model_path = PROJECT_ROOT / "data" / "processed" / "model.pkl"
preds_path = PROJECT_ROOT / "data" / "processed" / "predictions_latest.csv"
preds_path.parent.mkdir(parents=True, exist_ok=True)

save_model(result, model_path)

preds_for_app = result.test_predictions.merge(
    feature_table[["Player","season","Pos_primary","Age","MP"]],
    on=["Player","season"], how="left"
)
preds_for_app.to_csv(preds_path, index=False)

print(f"Model saved:       {model_path}")
print(f"Predictions saved: {preds_path}")
print(f"   ({len(preds_for_app)} player predictions written)")
print()
print("Launch the dashboard:")
print("    streamlit run src/app/dashboard.py")

350 predictions (the full 2025 season) are written to CSV for the dashboard. The model pickle contains the trained LightGBM booster, hyperparameters, test metrics, and the feature name list — everything needed to reproduce predictions or score new player stats without retraining.

---
## 8. Scoring New Players

The scoring module (`src/model/score.py`) is kept separate from training code.
Inference has no dependency on Optuna, sklearn metrics, or the training loop —
the kind of separation that matters when deploying to a lightweight API or
a notebook that doesn't need to retrain.

Three entry points:
- `load_model(path)` — load the saved pickle bundle
- `score_players(bundle, feature_table)` — score a full feature table, returns ranked DataFrame
- `score_single_player(bundle, stats)` — score one player from a flat stats dict

In [ ]:
from src.model.score import load_model, score_players, score_single_player

# Load the model we just saved
bundle = load_model(model_path)
print(f"Model loaded. Test R² from training: {bundle['test_r2']:.3f}")
print(f"Features expected: {len(bundle['features'])}")
print()

# Score the full feature table
ranked = score_players(bundle, feature_table, id_cols=["Player", "season", "Pos_primary", "Age", "MP"])
print("Top 5 predicted salaries:")
print(ranked[["Player", "Pos_primary", "Age", "MP", "predicted_usd", "actual_usd", "residual_usd"]].head(5).to_string(index=False))
print()

# Score a single hypothetical player
hypothetical = {
    "Age": 26, "Age_sq": 676, "is_prime": 1,
    "G": 70, "GS": 65, "MP": 33.0,
    "PTS_per36": 22.0, "TRB_per36": 5.5, "AST_per36": 6.0,
    "STL_per36": 1.2, "BLK_per36": 0.4, "TOV_per36": 2.8,
    "FGA_per36": 17.5, "3PA_per36": 6.0, "FTA_per36": 4.5,
    "FG%": 0.47, "3P%": 0.38, "FT%": 0.84, "eFG%": 0.584,
    "PER": 21.0, "TS%": 0.605, "USG%": 26.0,
    "WS": 8.5, "WS/48": 0.178, "BPM": 3.2, "OBPM": 2.1, "DBPM": 1.1, "VORP": 3.8,
    "is_starter": 1, "is_high_usage": 1, "is_rotation_only": 0,
    "pos_PG": 0, "pos_SG": 1, "pos_SF": 0, "pos_PF": 0, "pos_C": 0,
}
est = score_single_player(bundle, hypothetical)
print("Hypothetical: 26yo SG, 22 PPG, 8.5 Win Shares, 3.8 VORP")
print(f"  Predicted salary:  ${est['predicted_usd']:,}")
print(f"  Predicted ($M):    ${est['predicted_m']}M")
print(f"  Contract tier:     {est['tier']}")
print(f"  % of 2025 cap:     {est['cap_pct_2025']}%")

The single-player estimate is calibrated against the synthetic training distribution. On real NBA data — where star WS and VORP values correlate more strongly with max-tier contracts — the same profile would predict higher. The scoring function is correct; the calibration reflects the training data it was applied against.

---
### SHAP — Single Player Explanation

The waterfall plot shows exactly how the model arrives at the salary estimate for the hypothetical player, starting from the baseline (expected log salary across all training players) and showing each feature's contribution.

In [ ]:
from src.model.score import _align_features

X_hyp_aligned = _align_features(bundle, pd.DataFrame([hypothetical]))
shap_hyp = explainer(X_hyp_aligned)
shap.plots.waterfall(shap_hyp[0], max_display=12)

**Reading the waterfall:** starting from E[f(x)] (the average model output across all training players), each bar shows a feature pushing the prediction up (red) or down (blue). The final value f(x) is the model's log-salary prediction for this player.

For the hypothetical 26yo SG (WS=8.5, VORP=3.8):
- **WS=8.5** is the largest upward driver — above average but not max-contract territory in the training distribution
- **is_prime=1** (age 26) and **VORP=3.8** add further upward pressure
- The prediction lands in the Role player tier ($9.86M) because the synthetic training data calibrates WS=8.5 below the star-contract threshold — on real NBA data the same profile would predict higher

This per-player breakdown is what separates SHAP from global feature importance: it answers not just "what matters in general" but "why did this specific player get this specific prediction."

---
## Summary

The full pipeline ran successfully:

| Step | Result |
|---|---|
| Data acquisition | 1,400 player-seasons across 4 seasons (350/season) |
| Feature engineering | 36 features from 58-column table |
| Model training | LightGBM, 90 rounds (early stopping) |
| Test R² | 0.741 (held-out 2025 season) |
| Test MAE | $1,390,130 per player |
| Top predictors | Win Shares, Age, VORP, PER, BPM |
| Predictions saved | 350 rows to `data/processed/predictions_latest.csv` |

**Common interview questions about this pipeline:**

| Question | Where to find the answer |
|---|---|
| Why LightGBM instead of XGBoost? | `src/model/train.py` module docstring |
| Why log-transform salary? | `src/pipeline/features.py` docstring, checkpoint 6 plot |
| How do you handle traded players? | `src/scraper/basketball_reference.py` — TOT row logic |
| How do you prevent data leakage? | `src/model/train.py` — `_time_split()` function |
| What does the model get wrong? | Checkpoint 9 scatter — players on pre-cap-spike contracts |
| Why is Win Shares the top feature? | It is the composite stat most directly cited in contract negotiations |